# 05 — M3 CRNN (Apratim Sharma)

Convolutional front-end into a bidirectional LSTM, trained on the **identical
10,697 clips and splits** as M1 and M2 (cached log-mels from `fmc-m2-mels`).

**What M3 tests.** M2 halves time and frequency four times and then
global-average-pools, so temporal order is gone before the classifier. M3
pools **frequency only** and hands all 1,001 frames to a BiLSTM in order. If
the generative artefacts evolve over time, M3 can exploit that; if they are
static, it should match M2. The test-time time-order ablation at the end
measures directly whether M3 uses temporal order at all.

Runs as a pushed kernel (Save & Run All). No secret is attached, so nothing
is uploaded from here: every result is written to `/kaggle/working`, which a
committed run preserves as the version output, and is pulled down with
`kaggle kernels output`. `m3_results.json` is written the moment training
finishes, before any optional analysis, so a later crash cannot lose it.

In [ ]:
# Environment and inputs.
import os, glob, json, time, math, copy
import numpy as np, pandas as pd, torch

print(torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| gpus:', torch.cuda.device_count(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
# Fail immediately rather than silently training on CPU for hours.
assert torch.cuda.is_available(), 'no GPU attached to this kernel'

def find_input(name, marker):
    """Inputs mount at /kaggle/input/datasets/<user>/<name>/ (note the
    datasets/<user>/ prefix). Fall back to a search so a mount-layout change
    shows up as a clear message instead of a FileNotFoundError deep in a cell."""
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{marker}', recursive=True)
    print(f'{name}: expected path missing, search found {hits}')
    assert hits, f'{marker} not found under /kaggle/input'
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'mels.npy')
MAN_DIR  = find_input('fmc-manifest', 'manifest.csv')
for d in [MELS_DIR, MAN_DIR]:
    print(d, '->', sorted(os.listdir(d)))

OUT = '/kaggle/working'

In [ ]:
# Load M2's cached mels and prove they are the same clips in the same splits.
# Nothing is recomputed and nothing is re-sampled.

# subset.csv was written by the M2 notebook in the SAME row order as mels.npy,
# so row i of the CSV describes mels[i]. It is the authoritative split
# assignment; the manifest is used only to cross-check it.
sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
# mmap_mode='r': the array (2.74 GB on disk) is paged in from the read-only
# input mount on demand instead of being copied into RAM up front. The OS page
# cache keeps it hot after the first epoch, and forked DataLoader workers share
# the mapping instead of each holding a copy.
M      = np.load(f'{MELS_DIR}/mels.npy', mmap_mode='r')
labels = np.load(f'{MELS_DIR}/labels.npy')
man    = pd.read_csv(f'{MAN_DIR}/manifest.csv', dtype={'ytid': str})

# Shape and dtype must match the shared front-end (locked decision 3):
# 128 mels x 1001 frames at hop 160 = exactly 10.0 s.
assert M.shape == (10697, 128, 1001) and M.dtype == np.float16, (M.shape, M.dtype)
assert len(sub) == len(M) == len(labels)
assert (sub.label.values == labels).all(), 'labels.npy and subset.csv disagree'

# Cross-check against the manifest: every ytid has exactly one split (the
# leakage control), and the subset agrees with it.
assert (man.groupby('ytid').split.nunique() == 1).all(), 'a ytid appears in more than one split'
ytid_split = man.groupby('ytid').split.first()
assert (sub.ytid.map(ytid_split) == sub.split).all(), 'subset split != manifest split'

# Re-prove ytid-disjointness on the exact rows M3 trains on.
S = {s: set(sub.ytid[sub.split == s]) for s in ['train', 'val', 'test']}
assert not (S['train'] & S['val']) and not (S['train'] & S['test']) and not (S['val'] & S['test'])

counts = sub.groupby(['split', 'label']).size()
print(counts)
# The counts in CLAUDE.md section 4 and in M2's printout.
assert counts.to_dict() == {('test', 0): 804, ('test', 1): 800,
                            ('train', 0): 3746, ('train', 1): 3745,
                            ('val', 0): 802, ('val', 1): 800}
print(sub[sub.label == 1].groupby(['split', 'generator']).size().unstack())

# Chunked so the NaN scan never materialises a 1.4 GB boolean array at once.
n_nan = sum(int(np.isnan(M[i:i + 500]).sum()) for i in range(0, len(M), 500))
print('NaNs in mels:', n_nan)
assert n_nan == 0
print('ALL INTEGRITY CHECKS PASSED')

In [ ]:
# Normalisation and loaders.
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

idx = {s: np.where((sub.split == s).values)[0] for s in ['train', 'val', 'test']}

# One scalar mean and std from the TRAIN split only (locked decision 4),
# computed exactly as M2 did. Using val/test statistics would leak.
tr_mean = float(M[idx['train']].astype(np.float32).mean())
tr_std  = float(M[idx['train']].astype(np.float32).std())
print('train mel mean/std:', round(tr_mean, 3), round(tr_std, 3))
# M2 printed -45.663 / 17.12 from the same data. If these differ, M3 is not
# seeing what M2 saw and the comparison table is invalid, so stop the run.
assert abs(tr_mean - (-45.663)) < 1e-3 and abs(tr_std - 17.12) < 1e-3

class MelDS(Dataset):
    """Identical to M2's dataset: one normalised (1, 128, 1001) mel per item.
    np.asarray copies the row out of the memory map before arithmetic."""
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        j = self.rows[i]
        x = (np.asarray(M[j], dtype=np.float32) - tr_mean) / tr_std
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(float(labels[j]))

# Seeded generator: the train shuffle order is fixed by SEED, not by whatever
# random state earlier cells consumed.
g_loader = torch.Generator().manual_seed(SEED)
loaders = {
    s: DataLoader(MelDS(idx[s]), batch_size=32, shuffle=(s == 'train'),
                  generator=g_loader if s == 'train' else None,
                  num_workers=2, pin_memory=True)
    for s in ['train', 'val', 'test']
}
xb, yb = next(iter(loaders['train']))
print('batch:', xb.shape, yb.shape, '| train steps/epoch:', len(loaders['train']))

In [ ]:
# M3: convolutional front-end -> bidirectional LSTM.
import torch.nn as nn

class M3CRNN(nn.Module):
    """Conv front-end -> BiLSTM over time -> mean over time -> one logit.

    Every pooling layer halves FREQUENCY only, so the LSTM receives all 1001
    frames (one per 10 ms hop) in order. That is the controlled difference
    from M2, which pools time away before its classifier.
    """
    def __init__(self, n_mels=128, hidden=128):
        super().__init__()
        def blk(i, o):
            # One 3x3 conv per block (M2 uses two). These maps keep full time
            # resolution rather than halving it, so a second conv would cost
            # several times M2's compute; one keeps total FLOPs close to M2's
            # so the comparison is about structure, not size.
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 1)))   # (freq, time): halve freq, keep time
        self.features = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        f_out = n_mels // 16                          # 128 -> 8 bins after four halvings
        # LSTM input per frame = 128 channels x 8 freq bins = 1024. Flattening
        # instead of averaging over frequency keeps WHERE in the spectrum a
        # feature fired, which matters because the suspected shortcut sits in a
        # specific band (near Nyquist, locked decision 6).
        # Bidirectional: a clip-level label has no causality constraint, so
        # each frame may use both past and future context.
        self.rnn = nn.LSTM(input_size=128 * f_out, hidden_size=hidden,
                           batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hidden, 1))  # dropout as M2

    def forward(self, x):                               # x: (B, 1, 128, 1001)
        z = self.features(x)                            # (B, 128, 8, 1001)
        b, c, f, t = z.shape
        z = z.permute(0, 3, 1, 2).reshape(b, t, c * f)  # (B, 1001, 1024): time is the sequence
        seq, _ = self.rnn(z)                            # (B, 1001, 256)
        # Mean over time of the LSTM OUTPUTS, not the final hidden state: after
        # 1001 steps the final state is dominated by the end of the clip. This
        # is not the same as M2's global pooling, because each output here is
        # already a function of the whole ordered sequence.
        return self.head(seq.mean(dim=1))               # (B, 1) logit

dev = 'cuda'   # one GPU even on T4 x2: train time stays comparable with M2's
               # single-GPU 984.6 s and the checkpoint has no DataParallel prefix
model = M3CRNN().to(dev)
n_params = sum(p.numel() for p in model.parameters())
print('params:', n_params, '| conv:', sum(p.numel() for p in model.features.parameters()),
      '| lstm:', sum(p.numel() for p in model.rnn.parameters()))
with torch.no_grad():
    print('features out:', model.features(xb[:2].to(dev)).shape,
          '| logits:', model(xb[:2].to(dev)).shape)

In [ ]:
# Training: warmup + cosine LR, fp16 autocast with GradScaler, grad clipping.
from sklearn.metrics import balanced_accuracy_score

EPOCHS, PEAK_LR, CLIP = 20, 1e-3, 5.0
steps_per_epoch = len(loaders['train'])
total_steps = EPOCHS * steps_per_epoch
warm_steps = steps_per_epoch                        # one epoch of warmup

def lr_lambda(step):
    """Multiplier on PEAK_LR, applied every batch.
    Linear warmup 0.1 -> 1.0 over epoch 1: BatchNorm statistics and LSTM gates
    start random, and a full 1e-3 step on them is what produced M2's 0.61
    first epoch. Then cosine 1.0 -> 0.01: large steps early to explore, small
    steps late so the weights settle instead of hopping between solutions (M2's
    0.97 -> 0.82 drop at epoch 10 happened at a fixed 1e-3)."""
    if step < warm_steps:
        return 0.1 + 0.9 * step / warm_steps
    prog = (step - warm_steps) / max(1, total_steps - warm_steps)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * prog))

# Same optimiser and weight decay as M2: the schedule is the intended change.
opt    = torch.optim.Adam(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
lossf  = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda')               # T4 is Turing: fp16 + loss scaling, no bf16

# No early stopping, unlike M2: under cosine decay the lowest-LR, most stable
# epochs come LAST, so patience-based stopping would cut exactly those off.
# Overfitting is still controlled by keeping the best-validation checkpoint.
best_val, best_ep, best_state, history = -1.0, -1, None, []
t0 = time.time()

for ep in range(EPOCHS):
    model.train(); run_loss = torch.zeros((), device=dev); n = 0
    for xb, yb in loaders['train']:
        xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            loss = lossf(model(xb).squeeze(1), yb)
        scaler.scale(loss).backward()
        # Unscale BEFORE clipping, otherwise the threshold would be compared
        # against gradients multiplied by the loss scale (~65k). Clipping
        # matters for LSTMs: backprop through 1001 steps can produce rare huge
        # gradients that undo an epoch of progress.
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(opt); scaler.update()
        sched.step()                                # per batch, not per epoch
        run_loss += loss.detach() * len(yb); n += len(yb)   # no .item(): avoids a GPU sync per step

    # Validation. Checkpoint selection uses PLAIN accuracy, the rule M1 and M2
    # used (locked decision 10): changing it for M3 alone would break the
    # common protocol. Balanced accuracy is logged alongside; on an 802/800
    # val split the two nearly coincide.
    model.eval(); vlog, vtrue, vloss = [], [], 0.0
    with torch.no_grad():
        for xb, yb in loaders['val']:
            xb, yb = xb.to(dev), yb.to(dev)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logit = model(xb).squeeze(1).float()
            vloss += lossf(logit, yb).item() * len(yb)
            vlog.append(logit.cpu()); vtrue.append(yb.cpu())
    vlog = torch.cat(vlog).numpy(); vtrue = torch.cat(vtrue).numpy().astype(int)
    vpred = (vlog > 0).astype(int)
    va    = float((vpred == vtrue).mean())                 # selection metric
    va_ba = float(balanced_accuracy_score(vtrue, vpred))   # logged only
    rec = {"epoch": ep, "train_loss": round(float(run_loss) / n, 4),
           "val_loss": round(vloss / len(vtrue), 4), "val_acc": round(va, 4),
           "val_bal_acc": round(va_ba, 4),
           "lr_end": float(opt.param_groups[0]['lr']), "elapsed_s": round(time.time() - t0, 1)}
    history.append(rec)
    print(f"epoch {ep:2d}  train_loss {rec['train_loss']:.4f}  val_loss {rec['val_loss']:.4f}  "
          f"val_acc {va:.4f}  val_bal_acc {va_ba:.4f}  lr {rec['lr_end']:.2e}  ({rec['elapsed_s']:.0f}s)", flush=True)
    # Written every epoch so that a run killed mid-training still leaves its curve.
    json.dump(history, open(f'{OUT}/m3_history.json', 'w'), indent=2)

    if va > best_val:
        best_val, best_ep = va, ep
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, f'{OUT}/m3_best.pt')

model.load_state_dict(best_state)
m3_time = time.time() - t0
print(f'\nbest val acc {best_val:.4f} at epoch {best_ep} | train time {m3_time:.1f}s')

In [ ]:
# Test metrics -> m3_results.json, immediately after training.
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)

def predict(loader, transform=None):
    """Sigmoid scores over a loader. `transform` is used only by the time-order
    ablation below; None means the normal, intact input."""
    model.eval(); scores, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(dev)
            if transform is not None:
                xb = transform(xb)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                s = torch.sigmoid(model(xb).squeeze(1).float())
            scores.append(s.cpu().numpy()); trues.append(yb.numpy())
    return np.concatenate(scores), np.concatenate(trues).astype(int)

def metrics(true, score):
    """Line-for-line the M1/M2 metric code: threshold 0.5, EER at the ROC point
    where false-accept (fpr) and false-reject (1 - tpr) rates are closest."""
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

score, true = predict(loaders['test'])
assert (true == labels[idx['test']]).all()          # test loader is unshuffled, rows align
m3 = metrics(true, score)
for k, v in m3.items(): print(f'{k:20s} {v:.4f}')

# Beyond the shared M1/M2 metrics: test recall per generator and specificity on
# real clips. A generator well below the others is the likely hard E1 fold.
pred = (score > 0.5).astype(int)
gen_test = sub.generator.values[idx['test']]
per_gen = {str(g): round(float((pred[(true == 1) & (gen_test == g)] == 1).mean()), 4)
           for g in sorted(set(gen_test[true == 1]))}
spec_real = round(float((pred[true == 0] == 0).mean()), 4)
print('recall per generator:', per_gen, '| specificity (real):', spec_real)

# Every number is a variable produced by this run (rule 8); nothing is typed in.
results = {"model": "M3_CRNN", "seed": SEED, "device": "cuda",
           "gpu": torch.cuda.get_device_name(0),
           "train_time_s": round(m3_time, 1), "best_val_acc": round(best_val, 4),
           "best_epoch": best_ep, "selection_metric": "val_accuracy",
           "best_val_bal_acc": history[best_ep]["val_bal_acc"],
           "params": n_params,
           **{k: round(v, 4) for k, v in m3.items()},
           "n_test": int(len(true)),
           "test_recall_per_generator": per_gen, "test_specificity_real": spec_real,
           "config": {"epochs": EPOCHS, "batch_size": 32, "optimizer": "Adam",
                      "peak_lr": PEAK_LR, "weight_decay": 1e-4,
                      "schedule": "linear warmup 1 epoch (0.1->1.0) + cosine to 0.01x, per step",
                      "grad_clip_norm": CLIP, "amp": "fp16 autocast + GradScaler",
                      "early_stopping": "none (best-val checkpoint)",
                      "norm_mean": tr_mean, "norm_std": tr_std,
                      "input": "fmc-m2-mels (identical 10,697 clips and splits as M1/M2)"},
           "history": history}
json.dump(results, open(f'{OUT}/m3_results.json', 'w'), indent=2)
print('wrote', f'{OUT}/m3_results.json', '|', sorted(os.listdir(OUT)))

In [ ]:
# Comparison table (from results files only) and training curve.
import matplotlib.pyplot as plt

srcs = [('M2', f'{MELS_DIR}/m2_results.json'), ('M3', f'{OUT}/m3_results.json')]
rows = {n: json.load(open(p)) for n, p in srcs if os.path.exists(p)}
cols = ['balanced_accuracy', 'precision', 'recall', 'f1', 'auc', 'eer', 'train_time_s', 'params']
print(pd.DataFrame(rows).T.reindex(columns=cols).to_string())

# Approximate 95% interval on test balanced accuracy (normal approximation).
# A gap between models smaller than this is not evidence of a real difference.
ba = results['balanced_accuracy']
print(f"\nM3 bal acc 95% CI approx ±{1.96 * np.sqrt(ba * (1 - ba) / results['n_test']):.4f}")

h = pd.DataFrame(results['history'])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch, h.val_acc, marker='o', label='accuracy (selection)')
ax[0].plot(h.epoch, h.val_bal_acc, marker='.', label='balanced accuracy')
ax[0].set_title('M3 validation'); ax[0].legend()
ax[0].set_xlabel('epoch'); ax[0].set_ylim(0.5, 1.0); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_loss, label='train'); ax[1].plot(h.epoch, h.val_loss, label='val')
ax[1].set_title('M3 loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/m3_training_curve.png', dpi=150); plt.show()

In [ ]:
# Test-time time-order ablation: does M3 actually USE temporal order?
# No retraining: the best checkpoint is evaluated on test clips whose frames
# have been reordered. If accuracy barely moves, the cue is static.
g_abl = torch.Generator().manual_seed(SEED)

def block_shuffle(xb, block=50):
    """Reorder 0.5 s blocks (50 frames x 10 ms). Order WITHIN each block is
    kept, so the convs still see natural local texture; only longer-range order
    is destroyed. This is the cleaner test of temporal reliance."""
    T = xb.shape[-1]; starts = list(range(0, T, block))
    order = torch.randperm(len(starts), generator=g_abl).tolist()
    perm = torch.cat([torch.arange(starts[k], min(starts[k] + block, T)) for k in order])
    return xb[..., perm.to(xb.device)]

def frame_shuffle(xb):
    """Reorder every frame. Destroys ALL temporal order, but also creates
    frame-to-frame jumps the convs never saw in training, so its drop
    OVERSTATES how much M3 relies on time."""
    perm = torch.randperm(xb.shape[-1], generator=g_abl)
    return xb[..., perm.to(xb.device)]

abl = {"intact": m3}
for name, fn in [('block_shuffle_50', block_shuffle), ('frame_shuffle', frame_shuffle)]:
    s, t = predict(loaders['test'], transform=fn)
    abl[name] = metrics(t, s)
abl_df = pd.DataFrame(abl).T
abl_df['delta_bal_acc'] = abl_df.balanced_accuracy - m3['balanced_accuracy']
print(abl_df.round(4).to_string())

json.dump({"model": "M3_CRNN", "seed": SEED, "split": "test", "n_test": int(results['n_test']),
           "note": "test-time time-order ablation on the best M3 checkpoint; no retraining",
           **{k: {m: round(v, 4) for m, v in d.items()} for k, d in abl.items()}},
          open(f'{OUT}/m3_time_ablation.json', 'w'), indent=2)
print(sorted(os.listdir(OUT)))